# Paper 2 — GPT-4o Revision-Detection Validation (ARR Tier 2.1)

**Runtime: CPU, seconds — with a human labeling step in the middle.**

Measures GPT-4o's precision and recall on the revision-vs-filler decision, which has never been checked against a human.

**Part 1** builds a blind 120-item workbook (60 GPT-4o accepts from `labels/`, 60 rejects from `rejects/`). The rejects pile is what makes recall measurable — the earlier labeling workbooks sampled only accepts and could measure neither precision nor recall.

**Then stop and label the workbook.** Do not open the hidden reference CSV until you are finished.

**Part 2** scores it. Precision needs no prevalence correction; recall does, and the script reports both so the size of the correction is visible.

No scope labels are read anywhere in this notebook.


In [ ]:
# ---- Google Drive -----------------------------------------------------------
from google.colab import drive
drive.mount("/content/gdrive")

## Part 1 — build the blind annotation export


In [ ]:
# =============================================================================
# GPT-4o REVISION-DETECTION VALIDATION — BLIND ANNOTATION EXPORT  (ARR Tier 2.1)
#
# WHY THIS EXISTS, AND WHY THE EXISTING HAND LABELS CANNOT BE REUSED
# The existing labeling workbooks sampled ONLY from events GPT-4o had already
# confirmed as revisions. They therefore cannot evaluate GPT-4o's yes/no
# decision at all:
#   - precision is unmeasurable, because every item was already an accept;
#   - recall is unmeasurable, because the items GPT-4o REJECTED appear nowhere
#     in those workbooks.
# Measuring recall requires labeling items from the rejects/ pile. That is the
# whole point of this export, and it is why this is new labeling rather than a
# re-analysis of existing labels.
#
# WHAT IS BEING JUDGED
# One binary question per item: did a genuine REVISION occur at the marked
# position, or is the marker discourse padding (FILLER)? This is a materially
# easier judgment than the local/global SCOPE judgment that produced kappa
# ~0.46, and should not be expected to inherit that instability -- but it is
# still a single-annotator validation, which bounds what it can establish. See
# the LIMITS section printed at the end.
#
# -----------------------------------------------------------------------------
# THE SCHEMA TRAP THIS SCRIPT EXISTS TO SURVIVE
# The two piles record positions against DIFFERENT reference texts:
#   labels/{model}/{pid}.json   revisions[].char_start/char_end  -> indices into
#                               trace["full_response"]
#   rejects/{model}_confirmed_filler.json  word_idx (and sometimes char_start)
#                               -> indices into get_analysis_text(trace), i.e.
#                               clean_text(think_content) or response_clean
# Applying one convention to both piles silently highlights the WRONG SPAN for
# one of them, and the annotator would then be labeling text that has nothing
# to do with the recorded marker. Every item here is resolved against its own
# pile's reference text and then VERIFIED: the extracted span must equal the
# recorded matched_text. Items that fail verification are DROPPED and counted,
# never shipped. An unverified highlight is worse than a missing item.
#
# BLINDING
#   - items are shuffled before item_id is assigned, so the id encodes nothing
#   - no column states or implies the pile
#   - context is rendered with an identical character budget for both piles
#   - rejects-only fields (type_hint, filler_confidence) and labels-only fields
#     (confidence, reason) are kept OUT of the workbook entirely
#   - the hidden reference is written to a SEPARATE csv. Do not open it until
#     labeling is complete.
#
# OUTPUTS (to hand_labeling_export/)
#   gpt4o_validation_labeling.xlsx         <- label in this
#   gpt4o_validation_hidden_reference.csv  <- DO NOT OPEN until done
#   gpt4o_validation_pile_sizes.json       <- for prevalence correction
#
# Then run score_gpt4o_validation.py.
#
# COST: CPU only, seconds. No models loaded.
# =============================================================================



from pathlib import Path
from collections import defaultdict, Counter
import json, csv, random, re

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
LABELS_DIR  = GDRIVE_BASE / "labels"
REJECT_DIR  = GDRIVE_BASE / "rejects"
TRACES_DIR  = GDRIVE_BASE / "traces"
EXPORT_DIR  = GDRIVE_BASE / "hand_labeling_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ---- sampling parameters (PRE-REGISTER THESE; do not tune after looking) ----
N_PER_PILE   = 60      # 60 accepted + 60 rejected = 120 items
SEED         = 20261001
CONTEXT_CHARS = 700    # per side, before boundary snapping

OUT_XLSX   = EXPORT_DIR / "gpt4o_validation_labeling.xlsx"
OUT_HIDDEN = EXPORT_DIR / "gpt4o_validation_hidden_reference.csv"
OUT_SIZES  = EXPORT_DIR / "gpt4o_validation_pile_sizes.json"

MARK_OPEN, MARK_CLOSE = "⟦", "⟧"   # mathematical white square brackets


# =============================================================================
# reference-text reconstruction -- must match the original pipelines exactly
# =============================================================================
def clean_text(t):
    """Verbatim from task6_phase3_extract_filler.py. Do not 'improve' this --
    it must reproduce the exact string the rejects offsets were taken against."""
    return t.replace("Ġ", " ").replace("Ċ", "\n").replace("ĉ", "\t")


def get_analysis_text(tr):
    """Verbatim from task6_phase3_extract_filler.py -- the reference text for
    the REJECTS pile."""
    th = clean_text(tr.get("think_content", ""))
    cl = clean_text(tr.get("response_clean", ""))
    return th if th.strip() else cl

In [ ]:
def full_response_text(tr):
    """Reference text for the LABELS pile (see select_pilot_events.py, which
    slices full_response with revisions[].char_start)."""
    return tr.get("full_response", "")


def word_idx_to_char(text, widx):
    """Char offset of the widx-th whitespace-separated word. The rejects pile
    records word_idx; the original extractor converted it to a TOKEN index via
    the tokenizer, which is not reversible to characters here, so this
    recomputes the character position under the same whitespace convention
    (' '.join(text.split()[:widx])). Returns None if widx is out of range."""
    if widx is None or widx < 0:
        return None
    # Walk the ORIGINAL text, preserving real offsets, rather than assuming
    # text.split() rejoins with single spaces -- it does not, and assuming so
    # shifts every offset after the first run of repeated whitespace.
    for seen, m in enumerate(re.finditer(r"\S+", text)):
        if seen == widx:
            return m.start()
    return None


def resolve_span(ref_text, matched_text, char_start=None, char_end=None,
                 word_idx=None):
    """Locate matched_text in ref_text, by index arithmetic first and search
    only as a last resort. Returns (start, end, how) or (None, None, reason).

    Order of attempts, most to least trustworthy:
      1. recorded char_start/char_end, if the slice equals matched_text
      2. recorded char_start + len(matched_text)
      3. word_idx -> char offset, if the slice there equals matched_text
      4. unique case-insensitive occurrence of matched_text in ref_text
    Anything else fails. A non-unique search hit is a FAILURE, not a guess:
    picking the first of several occurrences would silently mark the wrong
    position, which is exactly the error mode this function exists to prevent.
    """
    if not matched_text:
        return None, None, "no matched_text recorded"
    mt = matched_text
    n = len(mt)

    # 1. both offsets recorded and consistent
    if char_start is not None and char_end is not None:
        if 0 <= char_start < char_end <= len(ref_text):
            if ref_text[char_start:char_end] == mt:
                return char_start, char_end, "char_start+char_end (exact)"

    # 2. char_start only (or char_end disagreed)
    if char_start is not None and 0 <= char_start <= len(ref_text) - n:
        if ref_text[char_start:char_start + n] == mt:
            return char_start, char_start + n, "char_start+len (exact)"
        if ref_text[char_start:char_start + n].lower() == mt.lower():
            return char_start, char_start + n, "char_start+len (case-insensitive)"

    # 3. word_idx
    if word_idx is not None:
        wc = word_idx_to_char(ref_text, word_idx)
        if wc is not None and ref_text[wc:wc + n].lower() == mt.lower():
            return wc, wc + n, "word_idx (case-insensitive)"

    # 4. unique occurrence only
    hits = [m.start() for m in re.finditer(re.escape(mt), ref_text, re.I)]
    if len(hits) == 1:
        return hits[0], hits[0] + n, "unique search fallback"
    if len(hits) == 0:
        return None, None, "matched_text absent from reference text"
    return None, None, f"matched_text occurs {len(hits)}x -- ambiguous, dropped"


def render_context(ref_text, start, end):
    """Context window with the marker delimited. Identical budget for both
    piles so window shape cannot leak which pile an item came from. Snapped
    outward to whitespace so words are not cut mid-token."""
    lo = max(0, start - CONTEXT_CHARS)
    hi = min(len(ref_text), end + CONTEXT_CHARS)
    # snap to whitespace, outward, so we never split a word
    while lo > 0 and not ref_text[lo - 1].isspace():
        lo -= 1
    while hi < len(ref_text) and not ref_text[hi].isspace():
        hi += 1
    pre  = ref_text[lo:start]
    mark = ref_text[start:end]
    post = ref_text[end:hi]
    ell_pre  = "[...] " if lo > 0 else ""
    ell_post = " [...]" if hi < len(ref_text) else ""
    body = f"{ell_pre}{pre}{MARK_OPEN}{mark}{MARK_CLOSE}{post}{ell_post}"
    return re.sub(r"\n{3,}", "\n\n", body).strip()


# =============================================================================
# load both piles
# =============================================================================
print("Indexing traces ...")
trace_index = {}
for md in sorted(TRACES_DIR.iterdir()):
    if md.is_dir():
        for f in md.glob("*.json"):
            trace_index[(md.name, f.stem)] = f
print(f"  {len(trace_index)} traces")

ref_cache = {}


def refs_for(model, pid):
    """(full_response, analysis_text) for one trace, cached."""
    key = (model, pid)
    if key in ref_cache:
        return ref_cache[key]
    p = trace_index.get(key)
    if p is None:
        ref_cache[key] = (None, None)
        return ref_cache[key]
    with open(p) as fh:
        tr = json.load(fh)
    ref_cache[key] = (full_response_text(tr), get_analysis_text(tr))
    return ref_cache[key]


# ---- pile A: GPT-4o ACCEPTED (labels/) --------------------------------------
print("\nLoading ACCEPTED pile (labels/) ...")
accepted = []
for md in sorted(LABELS_DIR.iterdir()):
    if not md.is_dir():
        continue
    for f in md.glob("*.json"):
        with open(f) as fh:
            obj = json.load(fh)
        pid = obj.get("problem_id", f.stem)
        for ri, rv in enumerate(obj.get("revisions", [])):
            if not rv.get("verified", True):
                continue
            accepted.append({
                "pile": "accepted", "model": md.name, "problem_id": pid,
                "entry_idx": ri,
                "matched_text": rv.get("matched_text"),
                "char_start": rv.get("char_start"),
                "char_end": rv.get("char_end"),
                "word_idx": rv.get("word_idx"),
                "marker": rv.get("marker"),
                # kept for the HIDDEN file only, never the workbook
                "gpt4o_confidence": rv.get("confidence"),
                "gpt4o_reason": rv.get("reason"),
            })
print(f"  {len(accepted)} verified revision entries")

# ---- pile B: GPT-4o REJECTED (rejects/) -------------------------------------
print("\nLoading REJECTED pile (rejects/) ...")
rejected = []
for f in sorted(REJECT_DIR.glob("*_confirmed_filler.json")):
    model = f.name.replace("_confirmed_filler.json", "")
    with open(f) as fh:
        entries = json.load(fh)
    for ei, e in enumerate(entries):
        rejected.append({
            "pile": "rejected", "model": model,
            "problem_id": e.get("problem_id"), "entry_idx": ei,
            "matched_text": e.get("matched_text"),
            "char_start": e.get("char_start"),
            "char_end": e.get("char_end"),
            "word_idx": e.get("word_idx"),
            "marker": e.get("marker"),
            "gpt4o_confidence": e.get("filler_confidence"),
            "gpt4o_reason": e.get("type_hint"),
        })
print(f"  {len(rejected)} confirmed-filler entries")

In [ ]:
PILE_SIZES = {"accepted_total": len(accepted), "rejected_total": len(rejected)}
if not accepted or not rejected:
    raise SystemExit(
        "\nFATAL: one pile is empty. Both are required -- the accepted pile "
        "measures precision, the rejected pile measures recall, and neither "
        "can substitute for the other.")


# =============================================================================
# resolve + verify every candidate BEFORE sampling
# =============================================================================
# Verification happens before the draw so the sample is drawn only from items
# whose highlight is known good. Verifying after the draw would shrink the
# sample unpredictably and make N depend on data quality.
print("\nResolving marker spans (index arithmetic, verified) ...")
resolved, drops = [], Counter()
for it in accepted + rejected:
    full, analysis = refs_for(it["model"], it["problem_id"])
    if full is None:
        drops["trace file missing"] += 1
        continue
    # pile-specific reference text -- THE critical distinction
    ref = full if it["pile"] == "accepted" else analysis
    ref_name = "full_response" if it["pile"] == "accepted" else "analysis_text"
    if not ref:
        drops[f"empty {ref_name}"] += 1
        continue

    s, e, how = resolve_span(ref, it["matched_text"], it.get("char_start"),
                             it.get("char_end"), it.get("word_idx"))
    if s is None:
        drops[how] += 1
        continue
    # paranoia: never ship a span we cannot read back as the recorded marker
    if ref[s:e].lower() != (it["matched_text"] or "").lower():
        drops["readback mismatch"] += 1
        continue

    it = dict(it)
    it.update({"ref_name": ref_name, "resolved_start": s, "resolved_end": e,
               "resolution_method": how,
               "context": render_context(ref, s, e)})
    resolved.append(it)

print(f"  resolved + verified: {len(resolved)}")
if drops:
    print("  dropped:")
    for k, v in drops.most_common():
        print(f"    {v:>5}  {k}")
by_method = Counter(x["resolution_method"] for x in resolved)
print("  resolution method:")
for k, v in by_method.most_common():
    print(f"    {v:>5}  {k}")
n_fallback = sum(v for k, v in by_method.items() if "search" in k)
if n_fallback:
    print(f"  NOTE {n_fallback} item(s) resolved by search fallback rather than "
          f"recorded offsets.\n       They are uniquely located and read back "
          f"correctly, so they are safe, but\n       a high count would mean the "
          f"recorded offsets are unreliable -- worth knowing.")

pool = defaultdict(list)
for it in resolved:
    pool[(it["pile"], it["model"])].append(it)

print("\nEligible pool by (pile, model):")
for k in sorted(pool):
    print(f"  {k[0]:<9} {k[1]:<34} {len(pool[k]):>6}")


# =============================================================================
# stratified draw: balance piles, and models within pile
# =============================================================================
rng = random.Random(SEED)
sample = []

In [ ]:
for pile in ("accepted", "rejected"):
    models = sorted(m for (p, m) in pool if p == pile)
    if not models:
        continue
    base, extra = divmod(N_PER_PILE, len(models))
    want = {m: base for m in models}
    for m in models[:extra]:
        want[m] += 1
    # redistribute from models that cannot fill their quota
    short = 0
    for m in models:
        have = len(pool[(pile, m)])
        if have < want[m]:
            short += want[m] - have
            want[m] = have
    while short > 0:
        progressed = False
        for m in models:
            if short <= 0:
                break
            spare = len(pool[(pile, m)]) - want[m]
            if spare > 0:
                take = min(spare, short)
                want[m] += take
                short -= take
                progressed = True
        if not progressed:
            print(f"  WARNING: {pile} pile short by {short} -- pool exhausted")
            break
    for m in models:
        sample += rng.sample(pool[(pile, m)], want[m])

print(f"\nDrawn: {len(sample)} items  "
      f"({sum(1 for x in sample if x['pile']=='accepted')} accepted / "
      f"{sum(1 for x in sample if x['pile']=='rejected')} rejected)")

# ---- BLINDING: shuffle, then assign ids, so id encodes nothing --------------
rng.shuffle(sample)
for i, it in enumerate(sample, start=1):
    it["item_id"] = i

# sanity: ids must not correlate with pile
acc_ids = [it["item_id"] for it in sample if it["pile"] == "accepted"]
assert len(set(acc_ids)) == len(acc_ids)
first_half = sum(1 for i in acc_ids if i <= len(sample) / 2)
print(f"Blinding check: accepted items in first half = {first_half}/"
      f"{len(acc_ids)} (should be ~{len(acc_ids)//2})")


# =============================================================================
# write the hidden reference FIRST, then the workbook
# =============================================================================
with open(OUT_HIDDEN, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["item_id", "pile", "gpt4o_verdict", "model", "problem_id",
                "entry_idx", "marker", "matched_text", "ref_name",
                "resolved_start", "resolved_end", "resolution_method",
                "gpt4o_confidence", "gpt4o_reason"])
    for it in sorted(sample, key=lambda x: x["item_id"]):
        w.writerow([
            it["item_id"], it["pile"],
            "revision" if it["pile"] == "accepted" else "filler",
            it["model"], it["problem_id"], it["entry_idx"], it.get("marker"),
            it["matched_text"], it["ref_name"], it["resolved_start"],
            it["resolved_end"], it["resolution_method"],
            it.get("gpt4o_confidence"), it.get("gpt4o_reason"),
        ])
print(f"\nWrote hidden reference: {OUT_HIDDEN}")
print("  *** DO NOT OPEN THIS FILE UNTIL LABELING IS COMPLETE ***")

with open(OUT_SIZES, "w") as f:
    json.dump({**PILE_SIZES,
               "accepted_eligible": sum(len(v) for k, v in pool.items() if k[0] == "accepted"),
               "rejected_eligible": sum(len(v) for k, v in pool.items() if k[0] == "rejected"),
               "accepted_sampled": len(acc_ids),
               "rejected_sampled": len(sample) - len(acc_ids),
               "seed": SEED, "n_per_pile": N_PER_PILE,
               "context_chars": CONTEXT_CHARS,
               "drops": dict(drops)}, f, indent=2)
print(f"Wrote pile sizes / sampling record: {OUT_SIZES}")


# =============================================================================
# workbook
# =============================================================================
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

TITLE = Font(bold=True, size=14)

In [ ]:
HEAD  = Font(bold=True, size=11, color="FFFFFF")
HFILL = PatternFill("solid", fgColor="31538F")
WRAP  = Alignment(wrap_text=True, vertical="top")
MONO  = Font(name="Consolas", size=10)
BOX   = Border(*(Side(style="thin", color="BBBBBB"),) * 4)

wb = Workbook()

# ---- instructions -----------------------------------------------------------
ws = wb.active
ws.title = "Instructions"
ws.column_dimensions["B"].width = 112
r = 2
ws.cell(r, 2, "GPT-4o Revision-Detection Validation — Blind Labeling").font = TITLE
r += 2
for block in [
    ("What you are doing", [
        "Each item shows a passage of a model's reasoning with ONE marker phrase",
        f"delimited like {MARK_OPEN}this{MARK_CLOSE}. Decide one thing: at that marker, did the model",
        "genuinely REVISE its approach, or was the marker just discourse padding (FILLER)?",
    ]),
    ("REVISION — mark this when", [
        "• the model abandons, replaces, or materially corrects a prior step or claim",
        "• it recomputes something it had already computed, because it doubted it",
        "• it notices an error, a contradiction, or a missed constraint and changes course",
        "The change must be visible in the text AFTER the marker. A stated intention to",
        "check, with no actual change, is not a revision.",
    ]),
    ("FILLER — mark this when", [
        "• the marker word appears but the reasoning continues on the same track",
        "• it is rhetorical emphasis, a transition, or thinking-aloud with no change",
        "• it restates or confirms what was just established rather than changing it",
        "• it introduces a NEW sub-step rather than revising an existing one",
    ]),
    ("UNCLEAR", [
        "Use it sparingly, and only when the passage genuinely does not settle the",
        "question — not for items you find hard. Unclear items are reported separately",
        "and are NOT silently dropped, so using it honestly costs nothing.",
    ]),
    ("Rules", [
        "1. Judge ONLY the delimited marker, not other revision-like language nearby.",
        "2. Judge from the shown context only. Do not open the traces.",
        "3. Do NOT open gpt4o_validation_hidden_reference.csv until you are finished.",
        "   It contains the answers; reading it invalidates the whole exercise.",
        "4. Items are shuffled and the item number tells you nothing about the answer.",
        "5. Fill in every row. Notes are optional and only used for disagreement review.",
    ]),
    ("Why this is being done", [
        "The pipeline used GPT-4o to decide which marker positions are real revisions.",
        "That decision has never been checked against a human. This measures its",
        "precision (how many accepts are real) and its recall (how many real revisions",
        "it rejected). Recall is why the rejected pile had to be included — the earlier",
        "labeling workbooks sampled only accepted items and could not measure it.",
    ]),
]:
    ws.cell(r, 2, block[0]).font = Font(bold=True, size=11)
    r += 1
    for line in block[1]:
        c = ws.cell(r, 2, line)
        c.alignment = WRAP
        r += 1
    r += 1

# ---- items ------------------------------------------------------------------
ws2 = wb.create_sheet("Items")
headers = ["item_id", "context (marker delimited)", "judgment", "notes (optional)"]
widths  = [9, 128, 16, 40]
for i, (h, wd) in enumerate(zip(headers, widths), start=1):
    c = ws2.cell(1, i, h)
    c.font, c.fill, c.alignment = HEAD, HFILL, Alignment(horizontal="center")
    ws2.column_dimensions[get_column_letter(i)].width = wd
ws2.freeze_panes = "A2"

In [ ]:
dv = DataValidation(type="list", formula1='"REVISION,FILLER,UNCLEAR"',
                    allow_blank=True, showDropDown=False)
dv.error = "Choose REVISION, FILLER, or UNCLEAR."
dv.errorTitle = "Invalid judgment"
ws2.add_data_validation(dv)

for row, it in enumerate(sorted(sample, key=lambda x: x["item_id"]), start=2):
    ws2.cell(row, 1, it["item_id"]).alignment = Alignment(horizontal="center",
                                                          vertical="top")
    c = ws2.cell(row, 2, it["context"])
    c.alignment, c.font, c.border = WRAP, MONO, BOX
    j = ws2.cell(row, 3, "")
    j.alignment = Alignment(horizontal="center", vertical="top")
    j.border = BOX
    dv.add(j)
    ws2.cell(row, 4, "").alignment = WRAP
    ws2.row_dimensions[row].height = 190

# ---- sampling record (no answers; safe to read) -----------------------------
ws3 = wb.create_sheet("Sampling record")
ws3.column_dimensions["A"].width = 46
ws3.column_dimensions["B"].width = 30
ws3.cell(1, 1, "Pre-registered sampling parameters").font = TITLE
rows = [
    ("", ""),
    ("random seed", SEED),
    ("target per pile", N_PER_PILE),
    ("context chars per side", CONTEXT_CHARS),
    ("items shipped", len(sample)),
    ("", ""),
    ("accepted-pile total (population)", PILE_SIZES["accepted_total"]),
    ("rejected-pile total (population)", PILE_SIZES["rejected_total"]),
    ("", ""),
    ("NOTE", "The two piles were sampled at DIFFERENT rates."),
    ("", "Precision/recall must therefore be reweighted to the"),
    ("", "true pile sizes; score_gpt4o_validation.py does this"),
    ("", "and reports raw in-sample figures alongside."),
]
for i, (k, v) in enumerate(rows, start=2):
    ws3.cell(i, 1, k).font = Font(bold=bool(k) and k == "NOTE")
    ws3.cell(i, 2, str(v))

wb.save(OUT_XLSX)
print(f"Wrote labeling workbook: {OUT_XLSX}")

print(f"""
{'='*74}
NEXT STEPS
{'='*74}
  1. Label all {len(sample)} rows in {OUT_XLSX.name} (Items sheet).
  2. Do NOT open {OUT_HIDDEN.name} before finishing.
  3. Run score_gpt4o_validation.py to get precision / recall / F1 with CIs.

LIMITS to state in the paper, not to discover in review
  - SINGLE ANNOTATOR. This measures agreement between GPT-4o and one human,
    which cannot be separated from that human's own idiosyncrasy. It bounds
    GPT-4o's reliability; it does not establish the construct. A second
    annotator on even a subset would let you report kappa alongside it.
  - The annotator is not independent of the pipeline's design, which is a
    conflict worth naming explicitly rather than leaving implicit.
  - Judged from a {CONTEXT_CHARS}-char window, not the full trace, so verdicts
    depend on revisions being locally visible. That is the same assumption the
    pipeline itself makes, so it is a shared limitation rather than a new one.
  - Prevalence: the sample is pile-balanced, NOT drawn at natural prevalence.
    Report the reweighted figures as primary.
{'='*74}""")

---

# ⏸ STOP — label the workbook now

1. Open `gpt4o_validation_labeling.xlsx` from `hand_labeling_export/` on your Drive.
2. Label all 120 rows on the **Items** sheet (REVISION / FILLER / UNCLEAR).
3. **Do not open** `gpt4o_validation_hidden_reference.csv` before finishing — it contains the answers.
4. Save the workbook back to the same path.

Then run Part 2 below. It refuses to score a partial sample.

---


## Part 2 — score it


In [ ]:
# =============================================================================
# SCORE THE GPT-4o REVISION-DETECTION VALIDATION  (ARR Tier 2.1, second half)
#
# Joins the completed blind workbook to the hidden reference and reports
# GPT-4o's precision and recall on the revision-vs-filler decision.
#
# -----------------------------------------------------------------------------
# THE ONE STATISTICAL POINT THAT MATTERS HERE
# The sample is PILE-BALANCED (60 accepted + 60 rejected), not drawn at the
# natural prevalence. That has asymmetric consequences:
#
#   PRECISION is unaffected. Precision conditions on GPT-4o having accepted,
#   and the accepted items were sampled at random within the accepted pile.
#   The in-sample figure is already an unbiased estimate.
#
#   RECALL is biased, and badly, unless the two piles happen to be the same
#   size. Recall = TP / (TP + FN), where TP comes from the accepted pile and
#   FN from the rejected pile. Taking the ratio inside a 50/50 sample
#   implicitly asserts the two piles are equal in the population. They are
#   not. Recall must be reweighted by the true pile sizes:
#
#       recall = (N_accepted * p_A) / (N_accepted * p_A + N_rejected * p_R)
#
#   where p_A = fraction of sampled ACCEPTED items the human calls REVISION
#         p_R = fraction of sampled REJECTED items the human calls REVISION
#
# Both figures are reported, with the reweighted one as primary and the naive
# in-sample one shown alongside so the size of the correction is visible.
#
# UNCLEAR is reported under three treatments rather than silently dropped,
# matching how unparseable rollouts are handled elsewhere in this project.
#
# USAGE:  python score_gpt4o_validation.py
# COST: CPU only, seconds.
# =============================================================================



from pathlib import Path
from collections import defaultdict, Counter
import csv, json, math, random

import numpy as np

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
EXPORT_DIR  = GDRIVE_BASE / "hand_labeling_export"

LABELED  = EXPORT_DIR / "gpt4o_validation_labeling.xlsx"
HIDDEN   = EXPORT_DIR / "gpt4o_validation_hidden_reference.csv"
SIZES    = EXPORT_DIR / "gpt4o_validation_pile_sizes.json"
OUT_JSON = EXPORT_DIR / "gpt4o_validation_results.json"

N_BOOT = 10000
SEED = 4242

VALID = {"REVISION", "FILLER", "UNCLEAR"}


# ----------------------------------------------------------------- helpers ---
def wilson(k, n, z=1.959964):
    """Wilson score interval. Used rather than the normal approximation
    because these proportions can sit near 0 or 1 at n=60, where the normal
    interval misbehaves (and can run outside [0,1])."""
    if n == 0:
        return (None, None)
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, centre - half), min(1.0, centre + half))


def f1(prec, rec):
    if prec is None or rec is None or (prec + rec) == 0:
        return None
    return 2 * prec * rec / (prec + rec)


# ------------------------------------------------------------------- load ---
for p in (LABELED, HIDDEN, SIZES):
    if not p.exists():
        raise SystemExit(f"FATAL: {p} not found. Run build_gpt4o_validation_export.py "
                         f"and complete the labeling first.")

In [ ]:
from openpyxl import load_workbook

wb = load_workbook(LABELED, data_only=True)
if "Items" not in wb.sheetnames:
    raise SystemExit("FATAL: workbook has no 'Items' sheet.")
ws = wb["Items"]

judgments, blank, invalid = {}, [], []
notes = {}
for row in ws.iter_rows(min_row=2, values_only=True):
    if not row or row[0] is None:
        continue
    iid = int(row[0])
    j = (row[2] or "").strip().upper()
    notes[iid] = (row[3] or "").strip() if len(row) > 3 else ""
    if not j:
        blank.append(iid)
        continue
    if j not in VALID:
        invalid.append((iid, j))
        continue
    judgments[iid] = j

hidden = {}
with open(HIDDEN) as f:
    for r in csv.DictReader(f):
        hidden[int(r["item_id"])] = r

sizes = json.load(open(SIZES))
N_A = sizes["accepted_total"]
N_R = sizes["rejected_total"]

print("=" * 74)
print("GPT-4o REVISION-DETECTION VALIDATION — RESULTS")
print("=" * 74)
print(f"\nitems in hidden reference : {len(hidden)}")

In [ ]:
print(f"items labeled             : {len(judgments)}")
if blank:
    print(f"  UNLABELED ({len(blank)}): {blank[:20]}{' ...' if len(blank) > 20 else ''}")
    print("  -> finish these before reporting anything; a partial sample is not"
          " the\n     pre-registered sample.")
if invalid:
    print(f"  INVALID ({len(invalid)}): {invalid[:10]}")
missing_ref = sorted(set(judgments) - set(hidden))
if missing_ref:
    raise SystemExit(f"FATAL: labeled item ids absent from hidden reference: "
                     f"{missing_ref[:10]} -- the files do not correspond.")

print(f"\npopulation pile sizes     : {N_A} accepted, {N_R} rejected")
print(f"sampling rates            : accepted {len([1 for i in judgments if hidden[i]['pile']=='accepted'])}/{N_A}"
      f" = {len([1 for i in judgments if hidden[i]['pile']=='accepted'])/N_A:.3%},"
      f"  rejected {len([1 for i in judgments if hidden[i]['pile']=='rejected'])}/{N_R}"
      f" = {len([1 for i in judgments if hidden[i]['pile']=='rejected'])/N_R:.3%}")
if abs(N_A - N_R) / max(N_A, N_R) > 0.1:
    print("  -> piles differ in size by >10%, so the naive in-sample recall IS")
    print("     materially biased. The reweighted figure is the one to report.")


# ------------------------------------------------------------- crosstab ----
print(f"\n{'='*74}\nCONFUSION MATRIX (human judgment x GPT-4o verdict)\n{'='*74}")
cross = Counter()
for iid, j in judgments.items():
    cross[(hidden[iid]["pile"], j)] += 1

print(f"  {'':<22}{'human REVISION':>16}{'human FILLER':>16}{'human UNCLEAR':>16}")
for pile, lab in (("accepted", "GPT-4o said REVISION"),
                  ("rejected", "GPT-4o said FILLER  ")):
    print(f"  {lab:<22}{cross[(pile,'REVISION')]:>16}"
          f"{cross[(pile,'FILLER')]:>16}{cross[(pile,'UNCLEAR')]:>16}")

n_unclear = cross[("accepted", "UNCLEAR")] + cross[("rejected", "UNCLEAR")]
print(f"\n  UNCLEAR total: {n_unclear}/{len(judgments)} "
      f"({n_unclear/max(1,len(judgments)):.1%})")


# ---------------------------------------------------- the three treatments --
# UNCLEAR is not missing-at-random: an item is unclear partly because the
# revision/filler boundary is genuinely fuzzy there, which is itself relevant
# to whether the construct is well defined. So it is reported three ways
# rather than dropped, and the conclusion has to survive the two defensible
# treatments. "Agrees with GPT-4o" is the generous bound, not a plausible
# reading, and is shown only as a stress test.
TREATMENTS = [
    ("excluded",        "UNCLEAR dropped (defensible: treat as missing)"),
    ("counts_against",  "UNCLEAR counts AGAINST GPT-4o (defensible: conservative)"),
    ("counts_for",      "UNCLEAR counts FOR GPT-4o (generous stress test)"),
]

In [ ]:
rng = random.Random(SEED)
results = {"pile_sizes": {"accepted": N_A, "rejected": N_R},
           "n_labeled": len(judgments), "n_unclear": n_unclear,
           "crosstab": {f"{k[0]}|{k[1]}": v for k, v in cross.items()},
           "treatments": {}}


def rates(mode):
    """Return (k_A, n_A, k_R, n_R) where k is the count judged REVISION.
    k_A/n_A -> p_A (precision). k_R/n_R -> p_R (miss rate within rejects)."""
    kA = cross[("accepted", "REVISION")]
    nA = cross[("accepted", "REVISION")] + cross[("accepted", "FILLER")]
    kR = cross[("rejected", "REVISION")]
    nR = cross[("rejected", "REVISION")] + cross[("rejected", "FILLER")]
    uA, uR = cross[("accepted", "UNCLEAR")], cross[("rejected", "UNCLEAR")]
    if mode == "excluded":
        pass
    elif mode == "counts_against":
        # against GPT-4o: unclear accepts are not revisions; unclear rejects are
        nA += uA
        kR += uR; nR += uR
    elif mode == "counts_for":
        # for GPT-4o: unclear accepts are revisions; unclear rejects are fillers
        kA += uA; nA += uA
        nR += uR
    return kA, nA, kR, nR


def reweighted_recall(kA, nA, kR, nR):
    """recall = N_A*p_A / (N_A*p_A + N_R*p_R)."""
    if nA == 0 or nR == 0:
        return None
    pA, pR = kA / nA, kR / nR
    tp, fn = N_A * pA, N_R * pR
    return tp / (tp + fn) if (tp + fn) > 0 else None


def boot_recall_ci(kA, nA, kR, nR, n_boot=N_BOOT):
    """Bootstrap CI for the reweighted recall. The two pile samples are
    resampled INDEPENDENTLY, matching how they were drawn.

    Resampling a Bernoulli sample and taking its mean is distributed as
    Binomial(n, p_hat)/n, so this draws from the binomial directly rather
    than looping over per-item choices -- identical distribution, and fast
    enough to afford N_BOOT=10000. A delta-method interval would also work
    but is less transparent about where the uncertainty comes from."""
    if nA == 0 or nR == 0:
        return (None, None)
    g = np.random.default_rng(SEED)
    pA = g.binomial(nA, kA / nA, size=n_boot) / nA
    pR = g.binomial(nR, kR / nR, size=n_boot) / nR
    tp, fn = N_A * pA, N_R * pR
    denom = tp + fn
    vals = np.divide(tp, denom, out=np.full_like(tp, np.nan), where=denom > 0)
    vals = vals[~np.isnan(vals)]
    if vals.size == 0:
        return (None, None)
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))


print(f"\n{'='*74}\nPRECISION / RECALL\n{'='*74}")
print("Precision needs no correction (it conditions on GPT-4o accepting, and")
print("accepts were sampled at random within their pile). Recall does.\n")

verdicts = {}
for mode, desc in TREATMENTS:
    kA, nA, kR, nR = rates(mode)
    if nA == 0 or nR == 0:
        print(f"  {desc}\n    insufficient data\n")
        continue
    pA, pR = kA / nA, kR / nR
    prec = pA
    prec_ci = wilson(kA, nA)
    rec_naive = pA / (pA + pR) if (pA + pR) > 0 else None
    rec = reweighted_recall(kA, nA, kR, nR)
    rec_ci = boot_recall_ci(kA, nA, kR, nR)
    the_f1 = f1(prec, rec)

    print(f"  {desc}")
    print(f"    precision        {prec:.3f}   95% CI [{prec_ci[0]:.3f}, {prec_ci[1]:.3f}]"
          f"   ({kA}/{nA} accepts judged real)")
    ci_txt = (f"   95% CI [{rec_ci[0]:.3f}, {rec_ci[1]:.3f}]"
              if rec_ci[0] is not None else "")
    print(f"    recall PRIMARY   {rec:.3f}{ci_txt}   (reweighted to pile sizes)")
    print(f"    recall naive     {rec_naive:.3f}   BIASED, shown for contrast "
          f"— correction {rec - rec_naive:+.3f}")
    print(f"    F1               {the_f1:.3f}   (precision + reweighted recall)")
    print(f"    miss rate within rejects p_R = {pR:.3f} ({kR}/{nR})")
    print()

    verdicts[mode] = {"precision": prec, "precision_ci95": list(prec_ci),
                      "recall_reweighted": rec, "recall_ci95": list(rec_ci),
                      "recall_naive_insample": rec_naive, "f1": the_f1,
                      "p_A": pA, "p_R": pR,
                      "k_A": kA, "n_A": nA, "k_R": kR, "n_R": nR}
results["treatments"] = verdicts


# -------------------------------------------------------- per-model split ---
print(f"{'='*74}\nPER-MODEL (UNCLEAR excluded; small n, directional only)\n{'='*74}")
per = defaultdict(Counter)
for iid, j in judgments.items():
    per[hidden[iid]["model"]][(hidden[iid]["pile"], j)] += 1
print(f"  {'model':<34}{'precision':>12}{'p_R':>10}{'n':>8}")
per_model = {}

In [ ]:
for m in sorted(per):
    c = per[m]
    kA = c[("accepted", "REVISION")]
    nA = kA + c[("accepted", "FILLER")]
    kR = c[("rejected", "REVISION")]
    nR = kR + c[("rejected", "FILLER")]
    p = kA / nA if nA else float("nan")
    q = kR / nR if nR else float("nan")
    print(f"  {m:<34}{p:>12.3f}{q:>10.3f}{nA+nR:>8}")
    per_model[m] = {"precision": p if nA else None, "p_R": q if nR else None,
                    "n_accepted": nA, "n_rejected": nR}
results["per_model"] = per_model
print("\n  Per-model n is ~15 per pile. Do not read these as separate results;")
print("  they are a check that no single model dominates the pooled figure.")


# -------------------------------------------------------------------- read --
print(f"\n{'='*74}\nREAD\n{'='*74}")
DEFENSIBLE = ["excluded", "counts_against"]
got = [m for m in DEFENSIBLE if m in verdicts]
if not got:
    print("Insufficient labeled data to read.")
else:
    precs = [verdicts[m]["precision"] for m in got]
    recs = [verdicts[m]["recall_reweighted"] for m in got]
    print(f"Across the {len(got)} defensible UNCLEAR treatments:")
    print(f"  precision {min(precs):.3f} – {max(precs):.3f}")
    print(f"  recall    {min(recs):.3f} – {max(recs):.3f}")
    lo_p, lo_r = min(precs), min(recs)

    if lo_p >= 0.85 and lo_r >= 0.85:
        print("\nGPT-4o's revision/filler decision is reliable on both sides.")
        print("The revision label can be treated as a reasonable operationalization,")
        print("and the attenuation it introduces into downstream results is small.")
    elif lo_p >= 0.85 and lo_r < 0.85:
        print("\nPrecision is good, recall is not: the events GPT-4o ACCEPTED are")
        print("mostly real revisions, but it REJECTED a meaningful number of real")
        print("ones. Consequences, stated precisely:")
        print("  - Results computed ON accepted events remain interpretable; they")
        print("    just describe a conservative subset of revisions.")
        print("  - The FILLER arm is contaminated: if GPT-4o's rejects contain real")
        print("    revisions, the matched-filler control is comparing revisions")
        print("    against a mixture, which biases that comparison TOWARD THE NULL.")
        print("    The validity results are therefore conservative, not inflated —")
        print("    but say so explicitly rather than leaving it to a reviewer.")
    elif lo_p < 0.85 and lo_r >= 0.85:
        print("\nRecall is good, precision is not: GPT-4o catches most real")
        print("revisions but also accepts non-revisions. The accepted pile is")
        print("diluted, which attenuates any revision-specific signal and means")
        print("effect sizes computed on it are UNDERSTATED.")
    else:
        print("\nBoth precision and recall are weak. The revision label itself is")
        print("then doing less work than the paper assumes, and that has to be")
        print("stated as a limitation on the construct, not just on the numbers.")

    if "counts_for" in verdicts:
        s = verdicts["counts_for"]
        print(f"\nGenerous stress test (UNCLEAR counted for GPT-4o): precision "
              f"{s['precision']:.3f}, recall {s['recall_reweighted']:.3f}.")
        print("Reported for transparency; not a plausible reading.")

    if n_unclear / max(1, len(judgments)) > 0.15:
        print(f"\nNOTE {n_unclear/len(judgments):.0%} of items were UNCLEAR. At that")
        print("rate the boundary is genuinely fuzzy, and that belongs in the paper")
        print("as a statement about the construct — not only as a sensitivity row.")

print("\nSINGLE ANNOTATOR. These figures measure GPT-4o against one human and")
print("cannot separate GPT-4o's error from that annotator's idiosyncrasy. State")
print("this as a limitation; a second annotator on a subset would let you report")
print("kappa alongside precision and recall.")

json.dump(results, open(OUT_JSON, "w"), indent=2, default=str)
print(f"\nWrote {OUT_JSON}")